# 02 資料處理入門：從 CSV 到分析就緒的 DataFrame

松柏護理之家退伍軍人症群聚事件，280 筆個案名冊已彙整完成。
這堂課我們把 CSV 讀進 pandas，檢查品質，建立衍生變項，做出翼區侵襲率統計表。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
# --- Step 1: 讀入 line list ---
import pandas as pd

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
print(f"資料維度：{df.shape[0]} 筆 × {df.shape[1]} 欄")
df.head()

In [ ]:
# --- Step 2: 檢視資料結構 ---
df.info()

In [ ]:
# 數值欄位的統計摘要
df.describe()

In [ ]:
# --- Step 3: 日期轉換 ---
# line list 有 5 個日期欄位，讀入時是文字 (object)
# 必須轉成 datetime 才能做時間計算

date_cols = [
    "facility_admission_date",
    "symptom_onset_date",
    "hospitalization_date",
    "death_date",
    "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# 驗證轉換結果
df[date_cols].dtypes

In [ ]:
# --- Step 4: 建立衍生變項 ---

# 1) 年齡組
df["age_group"] = pd.cut(
    df["age"],
    bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

# 2) 共病數
comorbidity_cols = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd",
    "immunosuppressed",
]
df["n_comorbidities"] = df[comorbidity_cols].sum(axis=1)

# 3) 是否感染（二元變項）
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# 4) 發病到住院天數
df["onset_to_hosp_days"] = (
    df["hospitalization_date"] - df["symptom_onset_date"]
).dt.days

# 5) 流行病學週
df["epi_week"] = df["symptom_onset_date"].dt.isocalendar().week

# 檢視新增的欄位
df[["case_id", "age", "age_group", "n_comorbidities", "infected",
    "onset_to_hosp_days", "epi_week"]].head(10)

In [ ]:
# --- Step 5: 處理遺漏值 ---
# 未感染者的 symptom_onset_date 等欄位會是 NaT（結構性遺漏）
# 這不是資料錯誤，不需要填補

print("=== 各欄位遺漏值數量 ===")
missing = df.isnull().sum()
print(missing[missing > 0].to_string())

print(f"\n未感染者有 onset 日期的數量："
      f"{df.loc[df['infected'] == 0, 'symptom_onset_date'].notna().sum()}")
print("→ 0 表示結構性遺漏沒有問題")

In [ ]:
# --- Step 6: groupby 分組統計 ---
# 按 floor × wing 計算侵襲率

wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(residents=("case_id", "size"), infected=("infected", "sum"))
    .reset_index()
)
wing_stats["attack_rate"] = wing_stats["infected"] / wing_stats["residents"]
wing_stats["attack_rate_pct"] = (wing_stats["attack_rate"] * 100).round(1)

print("=== 各翼區侵襲率 ===")
print(wing_stats.to_string(index=False))

## 小結

這堂課你完成了 line list 資料處理的六大步驟：

1. **讀入** CSV → `pd.read_csv()`
2. **檢視** 結構 → `df.info()`, `df.describe()`
3. **轉換** 日期 → `pd.to_datetime()`
4. **衍生** 新變項 → `pd.cut()`, `.sum(axis=1)`, `.dt.days`
5. **確認** 遺漏值 → 結構性遺漏 vs 資料錯誤
6. **統計** 分組指標 → `groupby().agg()`

下一堂課我們用這份清理好的資料來畫圖——流行曲線、年齡分布、翼區比較、熱力圖、互動圖。